# Simulación de modelo en producción (LightGBM)

Este notebook simula el proceso de producción, para ello se utilizan los artifacts que se guardaron en el proceso de modelado

## Carga del artefacto (simulando un proceso de scoring en producción)

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score


ROOT_DIR = Path.cwd().parent if (Path.cwd().name == "03_model_selection") else Path.cwd()
# Necesario solo para que joblib pueda resolver la clase real de cada transformer al deserializar
# (p.ej. src.transformers_utils.prod_transformers.TargetEncodingCVTransformer). El propio "fit"
# de esas clases NO se vuelve a ejecutar -- solo se restaura el estado ya aprendido.
sys.path.insert(0, str(ROOT_DIR))

PIPELINE_PATH = ROOT_DIR / "03_model_selection" / "artifacts" / "LightGBM" / "pipeline.joblib"

prod_pipeline = joblib.load(PIPELINE_PATH)
print(f"Pipeline cargado desde: {PIPELINE_PATH}")
print(f"Steps: {[name for name, _ in prod_pipeline.steps]}")

Pipeline cargado desde: /Users/dpiedrahita/proyectos/DS_pro/03_model_selection/artifacts/LightGBM/pipeline.joblib
Steps: ['preprocessor', 'post_encode', 'clf']


## Prueba 1 — reproducibilidad: mismas transacciones, mismo resultado

Se cargan filas de `test.csv` **solo como si fueran transacciones nuevas llegando a la API** (no se usan como "test set" de evaluación aquí). Si el `Pipeline` reproduce el preprocesamiento correctamente, el AUC sobre estas filas debe coincidir con el 0.8850 ya reportado en `01_evaluate_models.ipynb` para LightGBM -- confirmando que no hay diferencia entre "evaluar en el notebook de entrenamiento" y "cargar el artefacto en otro proceso y scorear".

In [2]:

TARGET_COL = "fraude"
ID_COL = "k"

incoming_df = pd.read_csv(ROOT_DIR / "dataset" / "test.csv")
incoming_df["fecha"] = pd.to_datetime(incoming_df["fecha"])

# En producción esto serían las columnas crudas que llegan por la API (nunca "fraude", que es lo que
# se quiere predecir; "k" tampoco, es el identificador -- ver ID_COL en 00_train_models.ipynb).
X_incoming = incoming_df.drop(columns=[TARGET_COL, ID_COL])
y_incoming = incoming_df[TARGET_COL]  # se usa aquí solo para verificar reproducibilidad, no para predecir

scores = prod_pipeline.predict_proba(X_incoming)[:, 1]

print(f"Filas scoreadas: {len(X_incoming)}")
print(f"AUC reproducido: {roc_auc_score(y_incoming, scores):.4f}  (esperado: 0.8850)")

Filas scoreadas: 45019
AUC reproducido: 0.8850  (esperado: 0.8850)


/Users/dpiedrahita/proyectos/DS_pro/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Prueba 2 — una transacción nueva, a la vez (como llegaría a una API)

Se construye una sola transacción cruda (un dict, como llegaría en un request) y se scorea sola, sin batch. Se usan dos thresholds de referencia (ya calculados en `01_evaluate_models.ipynb`): 0.5 (fijo) y 0.612 (óptimo por $J$ para LightGBM).

In [3]:
nueva_transaccion = X_incoming.iloc[[0]].copy()  # una sola fila, tal como llegaría un request
print(nueva_transaccion.T)

                         0
a                        4
b                   0.7858
c                842559.99
d                      2.0
e                      0.0
f                     13.0
g                       AR
h                        4
j              cat_bfa223b
l                   3124.0
m                    341.0
n                        1
o                      NaN
p                        Y
fecha  2020-04-10 00:00:21
monto                94.52
score                   34


In [4]:
proba = prod_pipeline.predict_proba(nueva_transaccion)[0, 1]

TH_FIJO = 0.5
TH_OPTIMO_J = 0.612  # ver 01_evaluate_models.ipynb

print(f"\nP(fraude) = {proba:.4f}")
print(f"Decisión @ TH fijo (0.5):      {'FRAUDE' if proba >= TH_FIJO else 'LEGITIMA'}")
print(f"Decisión @ TH óptimo (0.612):  {'FRAUDE' if proba >= TH_OPTIMO_J else 'LEGITIMA'}")


P(fraude) = 0.0209
Decisión @ TH fijo (0.5):      LEGITIMA
Decisión @ TH óptimo (0.612):  LEGITIMA


/Users/dpiedrahita/proyectos/DS_pro/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


## Prueba 3 — robustez ante una categoría nunca vista en entrenamiento

In [5]:
transaccion_categoria_nueva = X_incoming.iloc[[0]].copy()
transaccion_categoria_nueva["j"] = "cat_NUNCA_VISTO_EN_TRAIN"
transaccion_categoria_nueva["g"] = "ZZ"  # país/categoría inexistente

try:
    proba_nueva_categoria = prod_pipeline.predict_proba(transaccion_categoria_nueva)[0, 1]
    print(f"No hubo error. P(fraude) con categorías nunca vistas = {proba_nueva_categoria:.4f}")
    print(f"P(fraude) con la categoría original (misma fila, j/g reales) = {proba:.4f}")
except Exception as e:
    print(f"FALLÓ: {type(e).__name__}: {e}")

No hubo error. P(fraude) con categorías nunca vistas = 0.0062
P(fraude) con la categoría original (misma fila, j/g reales) = 0.0209


/Users/dpiedrahita/proyectos/DS_pro/.venv/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
